In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X, y = data.data, data.target          # X: (569, 30), y: (569,) with values 0 or 1

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# standardize using TRAIN statistics only
mean, std = X_train.mean(axis=0), X_train.std(axis=0)
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

print(X_train.shape, X_test.shape, y_train.mean())

(455, 30) (114, 30) 0.6285714285714286


In [2]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def loss(p, y, w, lam=0.0):
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)
    data_loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    penalty =  lam / 2 * sum(w**2)               
    return data_loss + penalty

def predict_prob(X, w, b):
    return sigmoid(X @ w + b)

def gradients(X, y, w, b, lam = 0.0):
    n = len(y)
    p = predict_prob(X, w, b)
    error = p - y
    dw = 1 / n * (X.T @ error) + lam * w
    db = 1 / n * np.sum(error)
    return dw, db

In [3]:
def numerical_gradients(X, y, w, b, eps=1e-5, lam = 0.0):
    dw_num = np.zeros_like(w)

    for i in range(len(w)):
        w_plus, w_minus = w.copy(), w.copy()
        w_plus[i] += eps
        w_minus[i] -= eps
        dw_num[i] = (loss(predict_prob(X, w_plus, b), y, w_plus, lam) - loss(predict_prob(X, w_minus, b), y, w_minus, lam)) / (2 * eps)

    db_num = (loss(predict_prob(X, w, b + eps), y, w, lam) - loss(predict_prob(X, w, b - eps), y, w, lam)) / (2 * eps)

    return dw_num, db_num

In [4]:
rng = np.random.default_rng(0)
w_test = rng.normal(0, 0.1, size=30)
b_test = 0.3

dw, db = gradients(X_train, y_train, w_test, b_test, lam=0.5)
dw_num, db_num = numerical_gradients(X_train, y_train, w_test, b_test, lam=0.5)

print("max diff in dw:", np.abs(dw - dw_num).max())
print("diff in db:    ", abs(db - db_num))

max diff in dw: 1.50748302729653e-11
diff in db:     1.0968864705418468e-12


In [5]:
def predict_label(X, w, b):
    return (predict_prob(X, w, b) >= 0.5).astype(int)

def train(lam, lr=0.1, steps=2000):
    w = np.zeros(X_train.shape[1])
    b = 0.0
    for _ in range(steps):
        dw, db = gradients(X_train, y_train, w, b, lam)
        w -= dw
        b -= db
    return w, b

for lam in [0, 0.01, 0.1, 1]:
    w, b = train(lam)
    train_acc = np.mean(predict_label(X_train, w, b) == y_train)
    test_acc = np.mean(predict_label(X_test, w, b) == y_test)
    print(f"lam={lam:<5} |w|={np.linalg.norm(w):7.3f}  max|w|={np.abs(w).max():6.3f}  train={train_acc:.3f}  test={test_acc:.3f}")

lam=0     |w|=  7.881  max|w|= 3.160  train=0.989  test=0.974
lam=0.01  |w|=  2.354  max|w|= 0.790  train=0.985  test=0.982
lam=0.1   |w|=  1.104  max|w|= 0.327  train=0.969  test=0.982
lam=1     |w|=  0.313  max|w|= 0.127  train=0.629  test=0.640


In [6]:
from sklearn.linear_model import LogisticRegression

C = 1.0
lam = 1 / (C * len(y_train))

w_mine, b_mine = train(lam, lr=0.1, steps=20000)

clf = LogisticRegression(C=C, max_iter=5000).fit(X_train, y_train)

print("max |w_mine - w_sklearn|:", np.abs(w_mine - clf.coef_[0]).max())
print("|b_mine - b_sklearn|:    ", abs(b_mine - clf.intercept_[0]))
print("my first 5 weights:     ", w_mine[:5])
print("sklearn first 5 weights:", clf.coef_[0][:5])

max |w_mine - w_sklearn|: 0.010961572923128556
|b_mine - b_sklearn|:     0.002050524407611165
my first 5 weights:      [-0.42780884 -0.39385837 -0.38947578 -0.46431062 -0.06677704]
sklearn first 5 weights: [-0.43190368 -0.38732553 -0.39343248 -0.46521006 -0.07166728]
